# Requirement 21: Protected temporary data

> Avoid persistent plaintext storage. Establish where temporary files live and ensure they are destroyed.

**Covered by:** Sandbox

**How it is fulfilled:** Working files live on the sandbox's local disk only, with no shared volumes or snapshots. When the sandbox is deleted its disk goes with it, and a new sandbox starts clean.

In [1]:
import sys
sys.path.insert(0, "..")

In [2]:
from helpers import *
sandbox, session_id = new_session()
info = sandbox.get()
print("volumes/mounts:", getattr(info, "volumes", None) or getattr(info, "mounts", None) or "none")
check("no snapshot source", info.sources_ref.snapshot is None)
sandbox.write_file("/work/plaintext.txt", "CONFIDENTIAL-MARKER-" + session_id)
print(sandbox.exec("df -h /work | tail -1").stdout)
print("file lives on:", sandbox.exec("findmnt -no SOURCE,FSTYPE -T /work").stdout.strip())

volumes/mounts: none
PASS  no snapshot source


/dev/vdb         20G  7.0G   12G  38% /

file lives on: /dev/vdb ext4


In [3]:
marker = "CONFIDENTIAL-MARKER-" + session_id
cleanup(sandbox)
check("sandbox is gone", not list(orchestrator().list_sandboxes(labels={"session": session_id})))
# A new sandbox from the same disk does not contain the old file (indirect wipe evidence)
fresh, _ = new_session()
check("new sandbox has no trace of the old file", fresh.exec("ls /work/plaintext.txt 2>&1").stdout.find("No such file") >= 0)
check("marker text not found anywhere on the new disk", fresh.exec(f"grep -rl --exclude-dir=proc --exclude-dir=sys --exclude-dir=dev -F '{marker}' / 2>/dev/null | wc -l").stdout.strip() == "0")
cleanup(fresh)

PASS  sandbox is gone


PASS  new sandbox has no trace of the old file


PASS  marker text not found anywhere on the new disk


**Caveat:** Disk wipe on delete is shown only indirectly. A new sandbox starting clean is evidence, not proof that the old blocks were scrubbed. Ask Microsoft for a written statement if Contoso needs it.